In [2]:
%pip install langchain 
%pip install langchain-community
%pip install langchain-huggingface
%pip install langchain-core
%pip install sentence_transformers
%pip install langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 146.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 231.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 186.4 MB/s  0:00:00
  Attempting uninstall: requests-toolbelt━━━━━━━━━━━━━━━━━━━━━  9/19 [annotated-types]ol]
    Found existing installation: requests-toolbelt 0.9.1━━━━━━  9/19 [annotated-types]
    Not uninstalling requests-toolbelt at /apps/Arch/software/Python/3.10.4-GCCcore-11.3.0/lib/python3.10/site-packages, outside environment /mimer/NOBACKUP/groups/oovgen/ziyuan/wasp_env
    Can't uninstall 'requests-toolbelt'. No files were found to uninstall./19 [annotated-types]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [langchain]19 [langchain]prebuilt]t]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
poetry 1.1.13 requires packaging<21.0,>=20.4, but

In [3]:
!wget -P /mimer/NOBACKUP/groups/oovgen/ziyuan/wasp-nlp/a3/data https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json 

--2026-05-21 14:46:10--  https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2584787 (2.5M) [text/plain]
Saving to: ‘/mimer/NOBACKUP/groups/oovgen/ziyuan/wasp-nlp/a3/data/ori_pqal.json’

ori_pqal.json       100%[===================>]   2.46M  --.-KB/s    in 0.04s   

2026-05-21 14:46:10 (68.5 MB/s) - ‘/mimer/NOBACKUP/groups/oovgen/ziyuan/wasp-nlp/a3/data/ori_pqal.json’ saved [2584787/2584787]



In [1]:
import pandas as pd
tmp_data = pd.read_json("data/ori_pqal.json").T
# some labels have been defined as "maybe", only keep the yes/no answers
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({"abstract": tmp_data.apply(lambda row: (" ").join(row.CONTEXTS+[row.LONG_ANSWER]), axis=1),
             "year": tmp_data.YEAR})
questions = pd.DataFrame({"question": tmp_data.QUESTION,
             "year": tmp_data.YEAR,
             "gold_label": tmp_data.final_decision,
             "gold_context": tmp_data.LONG_ANSWER,
             "gold_document_id": documents.index})

In [2]:
documents.head()

,abstract,year
21645374,Programmed cell death (PCD) is the regulated d...,2011
16418930,Assessment of visual acuity depends on the opt...,2006
9488747,Apparent life-threatening events in infants ar...,1997
17208539,The transanal endorectal pull-through (TERPT) ...,2007
10808977,Telephone counseling and tailored print commun...,2000


In [3]:
questions.head()

,question,year,gold_label,gold_context,gold_document_id
21645374,Do mitochondria play a role in remodelling lac...,2011,yes,Results depicted mitochondrial dynamics in viv...,21645374
16418930,Landolt C and snellen e acuity: differences in...,2006,no,"Using the charts described, there was only a s...",16418930
9488747,"Syncope during bathing in infants, a pediatric...",1997,yes,"""Aquagenic maladies"" could be a pediatric form...",9488747
17208539,Are the long-term results of the transanal pul...,2007,no,Our long-term study showed significantly bette...,17208539
10808977,Can tailored interventions increase mammograph...,2000,yes,The effects of the intervention were most pron...,10808977


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
# model = AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."<|eot_id|>


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
)

metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(texts=documents.abstract.tolist(), metadatas=metadatas)

In [7]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="collection",
    embedding_function=embeddings,
)

In [8]:
vector_store.add_documents(documents=texts)

['9f115430-cc75-4f48-a943-0e65ed0ecfc7',
 'a1ee28a3-2576-4025-b6bc-199181c29c68',
 '5f1936d4-03e9-4dc3-91f3-48ca590d1643',
 '35eec631-2430-40b8-b50f-2e127b55611a',
 '5ec808a7-9749-4641-9e6d-766d7743b4d7',
 'e233c90a-0e98-42e4-8b03-466b7fdf50d2',
 'e3fdc190-5b71-46e4-beda-e69534363133',
 'b33f10a7-1af0-42f5-9114-f7cfc9a17c5b',
 '6abbacf9-d67c-4f66-aeb5-9cbaba44b53a',
 'a63bb76e-ecbc-4810-be4e-d84b391ee99c',
 '9ce8b616-78f8-4abf-b136-e81873d1b774',
 '8369000c-6264-441d-a34e-cb7e6a7fe4de',
 'a889659b-5d5f-4494-9bdd-1a4c49e9b57f',
 '2057f46d-3649-4506-babf-1f154f4d755a',
 '9a4405ec-98d4-48d2-9066-ebdd75f1da53',
 'e1fb972f-4d33-4aa1-b646-8177e2910a9f',
 '6c9fe3ae-45fc-4d59-8298-f2d29f24d5ef',
 '32dcde15-2822-4daf-ba8d-9e548c69907f',
 'b3b2fd9a-793e-4d9c-bfc2-67227f70188c',
 'cfff4e38-84ee-42ee-93dd-b55f623efd75',
 '80702525-c3d6-4843-a10d-44fb0e7665e0',
 '39ae698e-6013-4f72-bcaf-c819195fd140',
 'e1c0d234-f448-408e-8b92-ba6e5a3c621b',
 '9265abec-5532-43e4-927e-7c381b78f83b',
 '6794a4d7-727b-

In [9]:
results = vector_store.similarity_search_with_score(
    "What is programmed cell death?", k=3
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=1.118844] Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; [{'id': 21645374}]
* [SIM=1.307049] with regard to the following variables: age, sex, comorbidity, weight loss, laboratory test results, histological type, ECOG score, TNM staging, and 

In [10]:
from typing import Any
from langchain_core.documents import Document
from langchain.agents.middleware import AgentMiddleware, AgentState


class State(AgentState):
    context: list[Document]


class RetrieveDocumentsMiddleware(AgentMiddleware[State]):
    state_schema = State

    def __init__(self, vector_store):
        self.vector_store = vector_store

    def before_model(self, state: AgentState) -> dict[str, Any] | None:
        last_message = state["messages"][-1] # get the user input query
        retrieved_docs = self.vector_store.similarity_search(last_message.text)  # search for documents

        docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)  

        augmented_message_content = f"""
        You are a binary classifier.

        Task:
        Answer the question using ONLY the retrieved context.

        Instructions:
        - Do not use outside knowledge.
        - Do not repeat or quote the context.
        - If the context does not contain enough information, answer "No".
        - Reason less than 50 words. 
        - Output ONLY the following format once:
        
        Reason: (less than words)
        Answer: Yes/No

        === CONTEXT START ===
        {docs_content}
        === CONTEXT END ===

        === QUESTION ===
        {last_message.content}
        
        === ANSWER ===
        """
        return {
            "messages": [last_message.model_copy(update={"content": augmented_message_content})],
            "context": retrieved_docs,
        }



In [11]:
from langchain.agents import create_agent
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", cache_dir = '/mimer/NOBACKUP/groups/oovgen/ziyuan/wasp-nlp/ckpt')
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", cache_dir = '/mimer/NOBACKUP/groups/oovgen/ziyuan/wasp-nlp/ckpt')

# tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-0528-Qwen3-8B",  cache_dir = '/mimer/NOBACKUP/groups/oovgen/ziyuan/wasp-nlp/ckpt')
# model = AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-0528-Qwen3-8B",  cache_dir = '/mimer/NOBACKUP/groups/oovgen/ziyuan/wasp-nlp/ckpt')
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    # max_new_tokens=128,
    device=0 if DEVICE.startswith("cuda") else -1,
)

model_pipeline = HuggingFacePipeline(
    pipeline=pipe
)

agent = create_agent(
    model_pipeline,
    tools=[],
    middleware=[RetrieveDocumentsMiddleware(vector_store)],
)

# ── Baseline: LLM without retrieval (no RAG) ──────────────
# Create a baseline agent with NO middleware — pure LLM, no external knowledge

baseline_agent = create_agent(
    model_pipeline,
    tools=[],
    middleware=[],  # ← no retrieval middleware
)

# Baseline prompt template (no context)
def build_baseline_prompt(question: str) -> str:
    return f"""You are a binary classifier.

            Task:
            Answer the question using your own knowledge.

            Instructions:
            - Output ONLY the following format once:

            Answer: Yes/No

            === QUESTION ===
            {question}
            
            === ANSWER ===
            """



config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Using device: cuda:0


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [12]:
questions.question.iloc[10]

'Therapeutic anticoagulation in the trauma patient: is it safe?'

In [17]:
import re
from tqdm import tqdm
import pandas as pd
import traceback

def extract_answer(text: str) -> str | None:
    # 1. Match "Answer: Yes" / "Answer: No" (case-insensitive)
    match = re.search(r'Answer:\s*(Yes|No)', text, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()

    # 2. Take the last Yes/No occurrence in text
    matches = re.findall(r'\b(Yes|No)\b', text, re.IGNORECASE)
    if matches:
        return matches[-1].capitalize()

    return None


predictions = []
ground_truth = []
raw_outputs = []

baseline_predictions = []
baseline_raw_outputs = []

questions_subset = questions.head(50)  

results_df = questions_subset[["question", "gold_label"]].copy()
for idx, row in tqdm(questions_subset.iterrows(), total=len(questions_subset), desc="Evaluating"):
    query = row["question"]
    gold = row["gold_label"].capitalize()

    result = agent.invoke({"messages": [{"role": "user", "content": query}]})
    rag_output = result["messages"][-1].content
    rag_output_clean = rag_output.split('=== ANSWER ===')[-1]
    rag_pred = extract_answer(rag_output_clean)

    prompt = build_baseline_prompt(query)
    result = baseline_agent.invoke({"messages": [{"role": "user", "content": prompt}]})
    baseline_output = result["messages"][-1].content
    baseline_output_clean = baseline_output.split('=== ANSWER ===')[-1]
    baseline_pred = extract_answer(baseline_output_clean)

    if rag_pred is not None and baseline_pred is not None:
        ground_truth.append(gold)
        predictions.append(rag_pred)
        raw_outputs.append(rag_output_clean)
        baseline_predictions.append(baseline_pred)
        baseline_raw_outputs.append(baseline_output_clean)
    else:
        tqdm.write(f"Q{idx} extraction failed (RAG={rag_pred}, Baseline={baseline_pred}): {query[:80]}...")
        results_df.drop(idx, inplace=True)


results_df["prediction"] = predictions
results_df["raw_output"] = raw_outputs

print(f"\nTotal questions: {len(questions)}")
print(f"Evaluated (both succeeded): {len(predictions)}")
print(f"Dropped (extraction failed): {len(questions_subset) - len(predictions)}")

from collections import Counter


Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Evaluating:   2%|▏         | 1/50 [00:06<04:54,  6.01s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https

Q22694248 extraction failed (RAG=Yes, Baseline=None): Is there a model to teach and practice retroperitoneoscopic nephrectomy?...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Evaluating:  38%|███▊      | 19/50 [01:35<02:22,  4.60s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_

Q23177368 extraction failed (RAG=None, Baseline=No): Does immediate breast reconstruction compromise the delivery of adjuvant chemoth...

Total questions: 890
Evaluated (both succeeded): 48
Dropped (extraction failed): 2


In [19]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
print(classification_report(ground_truth, predictions, digits=4))

print(classification_report(ground_truth, baseline_predictions, digits=4))



              precision    recall  f1-score   support

          No     0.6154    0.5000    0.5517        16
         Yes     0.7714    0.8438    0.8060        32

    accuracy                         0.7292        48
   macro avg     0.6934    0.6719    0.6788        48
weighted avg     0.7194    0.7292    0.7212        48

              precision    recall  f1-score   support

          No     0.5000    0.5625    0.5294        16
         Yes     0.7667    0.7188    0.7419        32

    accuracy                         0.6667        48
   macro avg     0.6333    0.6406    0.6357        48
weighted avg     0.6778    0.6667    0.6711        48



In [20]:
# ── Retrieval Recall: is the gold document retrieved? ──────
k_values = [1, 3, 5, 10, 20]

recall_at_k = {k: 0 for k in k_values}
total = 0

for idx, row in tqdm(questions.iterrows(), total=len(questions), desc="Retrieval Eval"):
    query = row["question"]
    gold_doc_id = row["gold_document_id"]

    retrieved_docs = vector_store.similarity_search(query, k=max(k_values))
    retrieved_ids = [doc.metadata["id"] for doc in retrieved_docs]

    for k in k_values:
        if gold_doc_id in retrieved_ids[:k]:
            recall_at_k[k] += 1
    total += 1

print(f"\nTotal questions evaluated: {total}")

for k in k_values:
    print(f"Recall@{k:<3}: {recall_at_k[k]/total:.4f}  ({recall_at_k[k]}/{total})")


Retrieval Eval: 100%|██████████| 890/890 [00:10<00:00, 87.34it/s]


Total questions evaluated: 890
Recall@1  : 0.9831  (875/890)
Recall@3  : 0.9933  (884/890)
Recall@5  : 0.9933  (884/890)
Recall@10 : 0.9944  (885/890)
Recall@20 : 0.9966  (887/890)


In [21]:
# ── Detailed Inspection: spot-check retrieved docs & answers ─
n_samples = 5

print("=" * 70)
print("DETAILED INSPECTION (random samples)")
print("=" * 70)

# sampled = questions.sample(n_samples, random_state=42)

for i, (idx, row) in enumerate(questions_subset.iterrows()):
    query = row["question"]
    gold_label = row["gold_label"]
    gold_context = row["gold_context"]
    gold_doc_id = row["gold_document_id"]

    retrieved_docs = vector_store.similarity_search(query, k=4)
    retrieved_ids = [doc.metadata["id"] for doc in retrieved_docs]
    gold_retrieved = gold_doc_id in retrieved_ids

    # Get model prediction from results_df
    try:
        row_idx = results_df.index.get_loc(idx)
        pred = results_df.iloc[row_idx]["prediction"]
        raw_output = results_df.iloc[row_idx]["raw_output"]

        print(f"\n{'─' * 70}")
        print(f"[Sample {i+1}]  Q: {query}")
        print(f"  Gold Label: {gold_label}  |  Predicted: {pred}  |  {'✅' if gold_label.capitalize() == pred else '❌'}")
        print(f"  Gold Doc ID: {gold_doc_id}  |  Retrieved IDs: {retrieved_ids}  |  Gold Retrieved: {'✅' if gold_retrieved else '❌'}")
        print(f"  Gold Context (first 150 chars): {gold_context[:150]}...")
        print(f"\n  --- Top Retrieved Chunks ---")
        for j, doc in enumerate(retrieved_docs):
            match_mark = " ★ GOLD" if doc.metadata["id"] == gold_doc_id else ""
            print(f"  [{j+1}] Doc#{doc.metadata['id']}{match_mark}: {doc.page_content[:120]}...")
        print(f"\n  --- Model Raw Output (last 250 chars) ---")
        print(f"  ...{raw_output[-250:]}")
        print()
    except:
        continue


DETAILED INSPECTION (random samples)

──────────────────────────────────────────────────────────────────────
[Sample 1]  Q: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
  Gold Label: yes  |  Predicted: Yes  |  ✅
  Gold Doc ID: 21645374  |  Retrieved IDs: [21645374, 21645374, 21645374, 21645374]  |  Gold Retrieved: ✅
  Gold Context (first 150 chars): Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other or...

  --- Top Retrieved Chunks ---
  [1] Doc#21645374 ★ GOLD: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascarien...
  [2] Doc#21645374 ★ GOLD: first time, we have shown the feasibility for the use of CsA in a whole plant system. Overall, our findings implicate th...
  [3] Doc#21645374 ★ GOLD: transition pore (PTP) formation during PCD was indirectly examined via in vivo cyclospo